# 04 — Joins: column lineage из нескольких источников

Демонстрация column lineage через INNER + LEFT joins трёх таблиц. Каждая выходная колонка тянется из конкретного источника (своего датасета).

**Что построим:** `default.customer_orders_enriched` — широкая таблица с данными клиента, заказа и (фиктивно привязанного) продукта.

Prerequisite: прогнан `00_setup.ipynb`. **Restart Jupyter kernel** перед запуском.

In [1]:
try:
    spark.stop()
except NameError:
    pass

from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .appName('lineage_04_joins')
    .enableHiveSupport()
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
assert spark.sparkContext.appName == 'lineage_04_joins', \
    f'appName fallback to {spark.sparkContext.appName!r} — restart Jupyter kernel'
print('app name:', spark.sparkContext.appName)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/27 13:09:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/27 13:09:23 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.
26/05/27 13:09:31 WARN BlackbirdModule: Unable to find Java 9+ MethodHandles.privateLookupIn.  Blackbird is not performing optimally!
26/05/27 13:09:31 WARN SparkApplicationNameApplicationJobNameProvider: Failed to obtain the application name. Using the default value [unknown]. Set the [spark.app.name] or [spark.openlineage.appName] property.
26/05/27 13:09:32 WARN SparkApplicationNameApplicationJobNameProvider: Failed to obtain the application name. Using the default value [unknown]. Set the [spark.app.name] or [spark.openlineage.appName] property.


app name: lineage_04_joins


In [2]:
df = spark.sql('''
    SELECT
        o.order_id,
        c.id           AS customer_id,
        c.name         AS customer_name,
        c.country      AS customer_country,
        o.amount       AS order_amount,
        o.order_ts     AS order_ts,
        p.category     AS product_category,
        p.price        AS product_price
    FROM default.raw_orders o
    INNER JOIN default.raw_customers c
        ON c.id = o.customer_id
    LEFT JOIN default.raw_products p
        ON p.product_id = (o.order_id % 60)
''')
df.write.mode('overwrite').saveAsTable('default.customer_orders_enriched')
spark.table('default.customer_orders_enriched').show(5, truncate=False)

# Sanity: LEFT JOIN с key % 60 при product_id 0..49 → ~17% строк с NULL category/price.
null_pct = spark.sql('''
    SELECT round(100.0 * sum(CASE WHEN product_category IS NULL THEN 1 ELSE 0 END) / count(*), 1) AS null_pct
    FROM default.customer_orders_enriched
''').collect()[0]['null_pct']
print(f'product_category NULL rate: {null_pct}%  (ожидаем ~16.7%)')

26/05/27 13:09:35 WARN HiveConf: HiveConf of name hive.metastore.event.db.notification.api.auth does not exist
26/05/27 13:09:35 WARN HiveConf: HiveConf of name hive.log.dir does not exist
26/05/27 13:09:35 WARN HiveConf: HiveConf of name hive.root.logger does not exist
26/05/27 13:09:35 WARN HiveClientImpl: Detected HiveConf hive.execution.engine is 'tez' and will be reset to 'mr' to disable useless hive logic
26/05/27 13:09:40 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


+--------+-----------+-------------+----------------+------------------+-------------------+----------------+------------------+
|order_id|customer_id|customer_name|customer_country|order_amount      |order_ts           |product_category|product_price     |
+--------+-----------+-------------+----------------+------------------+-------------------+----------------+------------------+
|0       |0          |user_0       |RU              |619.189370225301  |2023-11-14 22:13:20|books           |28.188121484885876|
|1       |1          |user_1       |US              |509.60188424464815|2023-11-14 22:14:20|food            |45.99881903505633 |
|2       |2          |user_2       |DE              |832.5259388871524 |2023-11-14 22:15:20|tech            |76.90971666031955 |
|3       |3          |user_3       |FR              |263.22809041172354|2023-11-14 22:16:20|books           |66.31414615273215 |
|4       |4          |user_4       |RU              |670.2867696264135 |2023-11-14 22:17:20|food 

In [3]:
spark.stop()

## Что смотреть в Marquez

**Dataset-level lineage:**
- `default.customer_orders_enriched` имеет **3 input dataset'а** (`raw_customers`, `raw_orders`, `raw_products`).

**Column-level lineage (DIRECT):**
- Каждая выходная колонка указывает на свой источник: `customer_name` ← `raw_customers.name`, `product_category` ← `raw_products.category`, `order_amount` ← `raw_orders.amount` и т.д. — subtype `IDENTITY`.
- `customer_id` aliasит `c.id`, поэтому DIRECT edge пойдёт от `raw_customers.id`.

**Column-level lineage (INDIRECT/JOIN) — важный нюанс:**
> Per [OpenLineage Spark column-lineage docs](https://openlineage.io/docs/integrations/spark/spark_column_lineage): когда план содержит `Join` ноду, **каждая** выходная колонка получает дополнительные INDIRECT edges от **всех** join-key колонок (`c.id`, `o.customer_id`, `o.order_id`, `p.product_id`) с subtype `JOIN`. То есть в Marquez у любой выходной колонки будет видно больше входящих стрелок, чем "прямой" источник — это ожидаемо.

**LEFT JOIN:**
- Predicate `p.product_id = o.order_id % 60` при `product_id ∈ [0..49]` даёт ~16.7% строк с `NULL` в `product_category`/`product_price` — sanity-print в коде это подтверждает. Lineage всё равно сохраняется.